In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 6 — Ejercicio 1
# ---------------------------------------------------------------

import seaborn as sns
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

titanic = sns.load_dataset("titanic").dropna(subset=["embarked"])
cols_num = ["age", "fare", "sibsp", "parch"]
cols_cat = ["sex", "embarked", "class"]
X = titanic[cols_num + cols_cat]
y = titanic["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocesador = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputar",  SimpleImputer(strategy="median")),
        ("escalar",  MinMaxScaler()),   # ← cambiado
    ]), cols_num),
    ("cat", Pipeline([
        ("imputar",  SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore",
                                    sparse_output=False)),
    ]), cols_cat),
])

pipe = Pipeline([
    ("prep",   preprocesador),
    ("modelo", LogisticRegression(random_state=42, max_iter=1000)),
])
pipe.fit(X_train, y_train)
print(f"Accuracy con MinMaxScaler: {pipe.score(X_test, y_test):.4f}")


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 6 — Ejercicio 2
# ---------------------------------------------------------------

from sklearn.preprocessing import FunctionTransformer
import pandas as pd

def agregar_family_size(df):
    df = df.copy()
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    return df

cols_num_ext = ["age", "fare", "sibsp", "parch", "family_size"]

preprocesador_ext = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("imputar",  SimpleImputer(strategy="median")),
        ("escalar",  StandardScaler()),
    ]), cols_num_ext),
    ("cat", Pipeline([
        ("imputar",  SimpleImputer(strategy="most_frequent")),
        ("codificar", OneHotEncoder(handle_unknown="ignore",
                                    sparse_output=False)),
    ]), cols_cat),
])

pipe_ext = Pipeline([
    ("feature_eng", FunctionTransformer(agregar_family_size)),
    ("prep",        preprocesador_ext),
    ("modelo",      LogisticRegression(random_state=42, max_iter=1000)),
])
pipe_ext.fit(X_train, y_train)
print(f"Accuracy con FamilySize: {pipe_ext.score(X_test, y_test):.4f}")


In [ ]:
# ---------------------------------------------------------------
# CAPÍTULO 6 — Ejercicio 3
# ---------------------------------------------------------------

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Reemplazar LogisticRegression por RandomForest en el mismo pipeline
pipeline_rf = Pipeline([
    ('preprocesar', preprocesador),
    ('modelo', RandomForestClassifier(random_state=42))
])

param_grid = {
    'modelo__n_estimators': [50, 100, 200],
    'modelo__max_depth': [None, 5, 10],
    'preprocesar__num__imputar__strategy': ['mean', 'median'],
    'preprocesar__cat__imputar__strategy': ['most_frequent', 'constant']
}

gs = GridSearchCV(pipeline_rf, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
gs.fit(X_train, y_train)

print(f"Mejores params: {gs.best_params_}")
print(f"Mejor CV score: {gs.best_score_:.4f}")
print(f"Test accuracy:  {gs.score(X_test, y_test):.4f}")
